<a href="https://colab.research.google.com/github/Rosanwok/event-backend/blob/main/Batch_email_sending.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Configuration for Email Batches

In [ ]:
EXCEL_FILE = "/content/drive/MyDrive/email_list.xlsx"
EXCEL_SHEET_NAME = "Sheet1" # Change this to the exact sheet name in your Excel file
TEMPLATE_FILE = "/content/drive/MyDrive/Batch_Admission_letter_template.docx"
COMMON_DOCUMENT_FILE = "/content/drive/MyDrive/In Pursuit of Purpose (Myles Munroe).pdf"
CARDINAL_PILLARS_FILE = "/content/drive/MyDrive/HUST – Seven Cardinal Pillars.pdf"

In [ ]:
import pandas as pd

try:
    df_excel = pd.read_excel(EXCEL_FILE, sheet_name=EXCEL_SHEET_NAME)
    print(f"Successfully loaded data from '{EXCEL_FILE}' (sheet: '{EXCEL_SHEET_NAME}')")
    display(df_excel.head())
except FileNotFoundError:
    print(f"Error: The Excel file '{EXCEL_FILE}' was not found. Please verify the path.")
except Exception as e:
    print(f"An error occurred while reading the Excel file: {e}")

Successfully loaded data from '/content/drive/MyDrive/email_list.xlsx' (sheet: 'Sheet1')


,name,email,program
0,ABIJO Victoria Olawunmi,victoriaabijo9@gmail.com,Medicine and Surgery
1,ADEBAYO Abdulrosheed Bukunmi,adebayoabdulrosheed41@gmail.com,Medicine and Surgery
2,ADEOYE ABIGAEL ABIMBOLA,hollumide@yahoo.com,Medicine and Surgery
3,ADEYEYE Success Aduralere,adeyeyesuccessaduralere@gmail.com,Medicine and Surgery
4,ADJEKUGHELE Marvellous Oghene fejiro,Adjekughelem@gmail.com,Aerospace Engineering


In [ ]:
!apt-get update && apt-get install -y libreoffice
!pip install python-docx pandas openpyxl

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,620 kB]
Fetched 1,748 kB in 3s (541 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... D

In [ ]:
import os
import time
import subprocess
import smtplib
from email.message import EmailMessage
from getpass import getpass
import pandas as pd
import docx

# --- Configuration & File Paths ---
EXCEL_FILE = "/content/drive/MyDrive/Batch_Admission_Blasting.xlsx"
TEMPLATE_FILE = "Batch_Admission_letter_template.docx"
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 465

# Security Prompt for Gmail Credentials
SENDER_EMAIL = input("Enter your Gmail address: ").strip()
SENDER_PASSWORD = getpass("Enter your 16-character Gmail App Password: ").strip()

def replace_placeholders(doc, replacements):
    """Replaces placeholders in paragraphs and tables while preserving formatting."""
    for p in doc.paragraphs:
        for key, value in replacements.items():
            if key in p.text:
                for run in p.runs:
                    if key in run.text:
                        run.text = run.text.replace(key, str(value))

    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for p in cell.paragraphs:
                    for key, value in replacements.items():
                        if key in p.text:
                            for run in p.runs:
                                if key in run.text:
                                    run.text = run.text.replace(key, str(value))

def send_admission_email(recipient_email, recipient_name, pdf_path):
    """Sends customized email with the attached PDF admission letter."""
    msg = EmailMessage()
    msg['Subject'] = "Provisional Offer of Admission - Hillside University of Science & Technology"
    msg['From'] = SENDER_EMAIL
    msg['To'] = recipient_email

    body = f"""Dear {recipient_name},

Congratulations! We are pleased to inform you that you have been offered Provisional Admission into Hillside University of Science & Technology (HUST) for the 2026/2027 Academic Session.

Please find attached your official Admission Letter containing details regarding your program, acceptance procedures, and resumption requirements.

To accept this offer, please reply directly to this email.

Congratulations on your admission, and welcome to Hillside University of Science and Technology.

For further enquiry, please contact 09030402775, 09111229992 or 09119429029.

Warm regards,

Office of the Registrar
Hillside University of Science and Technology
Ikoro Road, Okemesi, Ekiti State, Nigeria

https://www.hust.edu.ng/
"""
    msg.set_content(body)

    # Attach the generated PDF
    with open(pdf_path, 'rb') as f:
        file_data = f.read()
        file_name = os.path.basename(pdf_path)

    msg.add_attachment(file_data, maintype='application', subtype='pdf', filename=file_name)

    # Send via Gmail SSL
    with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT) as server:
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.send_message(msg)

def process_batch_admissions():
    # Read recipient dataset
    df = pd.read_excel(EXCEL_FILE)

    total_records = len(df)
    print(f"Starting batch process for {total_records} candidates...\n")

    for index, row in df.iterrows():
        app_num = str(row['app_number']).strip()
        name = str(row['name']).strip()
        program = str(row['program']).strip()
        email = str(row['Email']).strip()

        print(f"[{index + 1}/{total_records}] Processing: {name} ({app_num})")

        # 1. Map template placeholders
        replacements = {
            "«app_number»": app_num,
            "«name»": name,
            "«program»": program
        }

        # 2. Customize DOCX
        doc = docx.Document(TEMPLATE_FILE)
        replace_placeholders(doc, replacements)

        temp_docx = f"Admission_Letter_{app_num}.docx"
        temp_pdf = f"Admission_Letter_{app_num}.pdf"
        doc.save(temp_docx)

        # 3. Convert DOCX to PDF using LibreOffice
        subprocess.run(
            ["libreoffice", "--headless", "--convert-to", "pdf", temp_docx],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

        # 4. Send Email with PDF Attachment
        try:
            send_admission_email(email, name, temp_pdf)
            print(f"   └─ Sent successfully to {email}")
        except Exception as e:
            print(f"   └─ Failed to send to {email}: {e}")

        # 5. Clean up temporary files
        if os.path.exists(temp_docx):
            os.remove(temp_docx)
        if os.path.exists(temp_pdf):
            os.remove(temp_pdf)

        # Pause to prevent email throttle limits
        time.sleep(2)

    print("\nBatch processing complete!")

if __name__ == "__main__":
    process_batch_admissions()

Enter your Gmail address: admission@hust.edu.ng
Enter your 16-character Gmail App Password: ··········
Starting batch process for 44 candidates...

[1/44] Processing: KUSIMO Simbiat Ajoke (3001)
   └─ Sent successfully to kusimosimbiat@gmail.com
[2/44] Processing: TOWOBA Adewunmi Mariam (3002)
   └─ Sent successfully to towobaadewunmimariam@gmail.com
[3/44] Processing: NWAOHA Michelle Chiagozie (3003)
   └─ Sent successfully to mimimichelle1974@gmail.com
[4/44] Processing: ADEKOYA Oluwasefunmi Marvelous (3004)
   └─ Sent successfully to adekoyaoluwasefunmi@gmail.com
[5/44] Processing: CHIDIADI Chukwudi David (3005)
   └─ Sent successfully to dchukwudi928@gmail.com
[6/44] Processing: SOSANYA Great Folagbade (3006)
   └─ Sent successfully to folagbadesosanya@gmail.com
[7/44] Processing: OCHIGBO Maria Gift (3007)
   └─ Sent successfully to ochigbomariagift@gmail.com
[8/44] Processing: SOLARU EMMANUEL AYOMIDE (3008)
   └─ Sent successfully to emmanuelsolaru18@gmail.com
[9/44] Processing: A

### Sending a Common Document with a Custom Message

This section provides code to send a common document (e.g., a book or a guide) to all admitted students with a predefined welcome message. This is distinct from the personalized admission letters sent previously.

In [ ]:
def send_common_document_email(recipient_email, recipient_name, book_path, pillars_path):
    """Sends a common email with attached documents to a recipient."""
    msg = EmailMessage()
    msg['Subject'] = "Your Journey Begins: A Welcome from Hillside University!"
    msg['From'] = SENDER_EMAIL
    msg['To'] = recipient_email

    body = f"""Dear {recipient_name},

Congratulations on your admission to Hillside University of Science and Technology (HUST), Okemesi-Ekiti. Welcome to the Hillside community. Every year, we watch students walk through our gates full of potential, and every year, the ones who thrive fastest share one thing in common: they arrive with the right mindset, not just the right grades. That is why, before you resume as a student of your program, we want to place something in your hands.

<strong> A Book Before Your First Class </strong>
Attached to this email is a copy of ‘’In Pursuit of Purpose (Myles Munroe)’’ a book we give to every incoming Hillside student. We don't ask you to read it for marks. We ask you to read it because how you think about goals, discipline, and growth in these next few weeks will shape how you experience the next phase of your life.
This is a small tradition, but it says something about how we operate at Hillside. Your development starts before your first lecture, not after it.

<strong> A Little About HUST </strong>
Our goal is simple: to deliver world-class education from right here in Nigeria. Everything we build is anchored on our Seven Cardinal Pillars: Purpose, People, Principles, Programs, Processes, Policies, and Performance, which you'll find detailed in the attached document. These pillars are not slogans; they are how we hold ourselves accountable to you and to every family that has placed its trust in us.

<strong> You're Already Part of Our Story </strong>
From today, you are an ambassador of Hillside. Share this journey with your family and friends, as someone in your circle may be weighing their own options, and your experience could be the reason they choose Hillside too.

Stay connected with us on Instagram and TIKTOK: hillsideuniversity

If you have any questions before resumption, reach our Admissions Team via email at admission@hust.edu.ng or bisola.bamidele@hust.edu.ng and via telephone on 09111229992 or 09030402775

We look forward to welcoming you to campus.
Warm regards,

<strong> Ven. Jibiya Jibrin Baros, FHRM </strong>
Senior Vice President/Dean of Students’ Development,
Hillside University of Science and Technology,
Okemesi, Ekiti State.

+234 803 812 8392
"""
    msg.set_content(body, subtype='html')

    # Attach the common documents
    for document_path in [book_path, pillars_path]:
        # Ensure the path is absolute and robustly checked
        abs_document_path = os.path.abspath(document_path)
        if not os.path.exists(abs_document_path):
            raise FileNotFoundError(f"Common document not found at: {abs_document_path}. Please check the path and file permissions.")

        with open(abs_document_path, 'rb') as f:
            file_data = f.read()
            file_name = os.path.basename(abs_document_path)

        msg.add_attachment(file_data, maintype='application', subtype='pdf', filename=file_name)

    # Send via Gmail SSL
    with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT) as server:
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.send_message(msg)

In [ ]:
print(os.listdir('/content/drive/MyDrive/'))

['Colab Notebooks', 'Uni_data.xlsx', 'email_message_nur.docx', 'All Uni_data.xlsx', 'email_message.docx', 'prospect.gsheet', 'prospect_.csv', 'all', 'filenames.csv', 'Google AI Studio', 'Batch_Admission_letter_template.docx', 'Batch_Admission_Blasting1.xlsx', 'Batch_Admission_Blasting.xlsx', 'In Pursuit of Purpose (Myles Munroe).pdf', 'HUST – Seven Cardinal Pillars.pdf', 'email_list.xlsx']


In [ ]:
process_common_document_emails()

Starting batch process for sending common documents to 44 candidates...

[1/44] Processing common document for: KUSIMO Simbiat Ajoke
   └─ Failed to send common document to kusimosimbiat@gmail.com: send_common_document_email() missing 1 required positional argument: 'pillars_path'
[2/44] Processing common document for: TOWOBA Adewunmi Mariam
   └─ Failed to send common document to towobaadewunmimariam@gmail.com: send_common_document_email() missing 1 required positional argument: 'pillars_path'
[3/44] Processing common document for: NWAOHA Michelle Chiagozie
   └─ Failed to send common document to mimimichelle1974@gmail.com: send_common_document_email() missing 1 required positional argument: 'pillars_path'
[4/44] Processing common document for: ADEKOYA Oluwasefunmi Marvelous
   └─ Failed to send common document to adekoyaoluwasefunmi@gmail.com: send_common_document_email() missing 1 required positional argument: 'pillars_path'


KeyboardInterrupt: 

In [ ]:
def process_common_document_emails():
    # Pre-check for required files
    for file_var, file_path in [('Excel File', EXCEL_FILE), ('Common Document (Myles Munroe)', COMMON_DOCUMENT_FILE), ('Cardinal Pillars Document', CARDINAL_PILLARS_FILE)]:
        abs_path = os.path.abspath(file_path)
        if not os.path.exists(abs_path):
            print(f"Error: {file_var} not found at: {abs_path}. Please verify the path in the configuration cell.")
            return # Stop execution if a critical file is missing

    # Read recipient dataset
    try:
        df = pd.read_excel(EXCEL_FILE, sheet_name=EXCEL_SHEET_NAME)
    except Exception as e:
        print(f"Error reading Excel file '{EXCEL_FILE}' (sheet: '{EXCEL_SHEET_NAME}'): {e}")
        return

    total_records = len(df)
    print(f"Starting batch process for sending common documents to {total_records} candidates...\n")

    for index, row in df.iterrows():
        name = str(row['name']).strip()
        email = str(row['email']).strip() # Corrected 'Email' to 'email'

        print(f"[{index + 1}/{total_records}] Processing common document for: {name}")

        # Send Email with common document Attachment
        try:
            send_common_document_email(email, name, COMMON_DOCUMENT_FILE, CARDINAL_PILLARS_FILE) # Updated call
            print(f"   └─ Common document sent successfully to {email}")
        except Exception as e:
            print(f"   └─ Failed to send common document to {email}: {e}")

        # Pause to prevent email throttle limits
        time.sleep(2)

    print("\nBatch processing for common documents complete!")

if __name__ == "__main__":
    process_common_document_emails()

Starting batch process for sending common documents to 40 candidates...

[1/40] Processing common document for: ABIJO Victoria Olawunmi
   └─ Common document sent successfully to victoriaabijo9@gmail.com
[2/40] Processing common document for: ADEBAYO Abdulrosheed Bukunmi
   └─ Common document sent successfully to adebayoabdulrosheed41@gmail.com
[3/40] Processing common document for: ADEOYE ABIGAEL ABIMBOLA
   └─ Common document sent successfully to hollumide@yahoo.com
[4/40] Processing common document for: ADEYEYE Success Aduralere
   └─ Common document sent successfully to adeyeyesuccessaduralere@gmail.com
[5/40] Processing common document for: ADJEKUGHELE Marvellous Oghene fejiro
   └─ Common document sent successfully to Adjekughelem@gmail.com
[6/40] Processing common document for: ADULOJU Dunmininu Promise
   └─ Common document sent successfully to pdunmininu285@gmail.com
[7/40] Processing common document for: AJEDE ABDULRAZAQ OLAJIDE
   └─ Common document sent successfully to abdu

In [ ]:
process_common_document_emails()

Starting batch process for sending common documents to 40 candidates...

[1/40] Processing common document for: ABIJO Victoria Olawunmi
   └─ Common document sent successfully to victoriaabijo9@gmail.com
[2/40] Processing common document for: ADEBAYO Abdulrosheed Bukunmi
   └─ Common document sent successfully to adebayoabdulrosheed41@gmail.com
[3/40] Processing common document for: ADEOYE ABIGAEL ABIMBOLA
   └─ Common document sent successfully to hollumide@yahoo.com
[4/40] Processing common document for: ADEYEYE Success Aduralere
   └─ Common document sent successfully to adeyeyesuccessaduralere@gmail.com
[5/40] Processing common document for: ADJEKUGHELE Marvellous Oghene fejiro
   └─ Common document sent successfully to Adjekughelem@gmail.com
[6/40] Processing common document for: ADULOJU Dunmininu Promise
   └─ Common document sent successfully to pdunmininu285@gmail.com
[7/40] Processing common document for: AJEDE ABDULRAZAQ OLAJIDE
   └─ Common document sent successfully to abdu

In [ ]:
import os
print(os.listdir('.'))

['.config', 'Batch_Admission_letter_template.docx', 'Batch_Admission_Blasting.xlsx', 'drive', 'sample_data']
